# 08 · Clean bedbug reports and evictions
**Outputs:** `data/processed/bedbugs_clean.parquet` and `data/processed/evictions_clean.parquet`

**Bedbugs:**
- Validate the BBL.
- Drop impossible filings: more infested units than units, negative counts, or no period.
- Keep one filing per building per year (the latest filed).

**Evictions:**
- Residential only, executed January 2025–October 2026, valid BBL.
- Drop duplicate records of the same case.

**Study window: January 1, 2025–October 31, 2026.** Annual comparisons should use January–October in both years. Housing prediction uses six-month histories and four-month outcomes; older figures and reports describe the previous study.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, housing as H, clean_housing as C
from src.preprocess import CleaningReport

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 70)

def show_report(rep):
    return pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]

def save(df, name, rep):
    df.to_parquet(C.OUT[name], index=False)
    (config.REPORTS_DIR / f"cleaning_report_{name}.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
    print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {C.OUT[name].relative_to(ROOT)}")

In [2]:
rep_bb = CleaningReport("bedbugs")
bedbugs = C.clean_bedbugs(H.load("bedbugs"), rep_bb)
display(show_report(rep_bb))
bedbugs.groupby("year").agg(buildings=("bbl", "size"), with_infestation=("has_infestation", "mean"),
                            infested_units=("infested_dwelling_unit_count", "sum"))

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,118012,118012,0,
1,valid_bbl,118012,117867,145,missing or invalid BBL
2,drop_impossible,117867,117859,8,"infested > units, negative counts, no period or units"
3,one_filing_per_building_year,117859,103089,14770,


,buildings,with_infestation,infested_units
year,,,
2025,103089,0.02,420428


In [3]:
save(bedbugs, "bedbugs", rep_bb)

Saved 103,089 rows × 10 columns → data\processed\bedbugs_clean.parquet


In [4]:
rep_ev = CleaningReport("evictions")
evictions = C.clean_evictions(H.load("evictions"), rep_ev)
display(show_report(rep_ev))
evictions[evictions["executed_date"].dt.month <= 10].groupby("year").size().to_frame("residential evictions")

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,34816,34816,0,
1,residential_only,34816,31905,2911,commercial evictions removed
2,parse_dates,31905,31905,0,missing or pre-2025 executed_date
3,valid_bbl,31905,29770,2135,missing or invalid BBL
4,drop_duplicates,29770,29762,8,"same case, docket, building and date"


,residential evictions
year,
2025,14396
2026,13193


In [5]:
save(evictions, "evictions", rep_ev)

Saved 29,762 rows × 5 columns → data\processed\evictions_clean.parquet


## Part C: PLUTO building data
Building traits (year built, apartments, floors, building class, coordinates) for every tax lot. `clean_pluto()` (in `src/preprocess.py`) does four things:
- Normalizes the BBL from `1000010010.00000000` to `1000010010`.
- Converts the numeric columns.
- Sets `yearbuilt = 0` to unknown.
- Validates ZIP codes and coordinates, and flags residential lots.

In [6]:
from src.preprocess import clean_pluto
pluto, rep_p = clean_pluto()
display(show_report(rep_p))
assert pluto["bbl"].is_unique and pluto["bbl"].str.fullmatch(r"[1-5]\d{9}").all()
pluto.to_parquet(config.CLEAN_PLUTO_FILE, index=False)
res = pluto[pluto.unitsres >= 3]
print(f"Saved {len(pluto):,} lots → {config.CLEAN_PLUTO_FILE.relative_to(ROOT)} | "
      f"buildings with 3+ apartments: {len(res):,} holding {res.unitsres.sum():,.0f} apartments")

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,858312,858312,0,
1,valid_unique_bbl,858312,858312,0,invalid or duplicate BBL
2,convert_types,858312,858312,0,"39,842 lots with yearbuilt=0 set to missing"


Saved 858,312 lots → data\processed\pluto_clean.parquet | buildings with 3+ apartments: 172,630 holding 2,873,922 apartments


All housing data is clean and keyed by BBL. → Next: **09_crime.ipynb**